# Module 04: Regularisation Paths

## Learning Objectives

By completing this notebook, you will:
1. Compute the full Lasso path using LARS on real data and visualise coefficient trajectories
2. Identify when Lasso fails on correlated features and verify ElasticNet succeeds
3. Select the optimal regularisation parameter using cross-validation
4. Interpret regularisation paths to rank feature stability

## Prerequisites
- Guide 01: Regularisation Paths (L1, L2, ElasticNet, LARS)
- Module 03: Wrapper methods
- scikit-learn linear models

## Estimated Time: 15 minutes

---

## 1. Setup

We use the California Housing dataset (regression) and Breast Cancer dataset (classification) throughout this notebook. Both are real, standard benchmarks with known feature relationships.

In [ ]:
# Imports — all standard scientific Python stack
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.datasets import fetch_california_housing, load_breast_cancer
from sklearn.linear_model import (
    lasso_path, enet_path,
    LassoCV, ElasticNetCV, Lasso, ElasticNet
)
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')
COLORS = plt.cm.tab20.colors

print("Imports complete.")

In [ ]:
# Load and prepare California Housing dataset
# This dataset has 8 features — small enough to visualise paths clearly
housing = fetch_california_housing(as_frame=True)
X_raw = housing.data
y_raw = housing.target

# Train/test split before any scaling (prevent data leakage)
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw, y_raw, test_size=0.2, random_state=42
)

# Standardise: Lasso penalty is scale-sensitive — ALWAYS standardise
scaler = StandardScaler()
X_train = pd.DataFrame(
    scaler.fit_transform(X_train_raw),
    columns=X_raw.columns
)
X_test = pd.DataFrame(
    scaler.transform(X_test_raw),
    columns=X_raw.columns
)

feature_names = list(X_raw.columns)
print(f"Features: {feature_names}")
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Target range: [{y_train.min():.2f}, {y_train.max():.2f}]")

## 2. Computing the Full Lasso Path with LARS

The LARS algorithm computes the complete regularisation path — all Lasso solutions from $\lambda_{\max}$ (empty model) to $\lambda \approx 0$ (full model) — in a single pass. This is equivalent in cost to fitting one OLS model.

The path is **piecewise linear**: between consecutive knots, each coefficient changes linearly in $\lambda$.

In [ ]:
# Compute Lasso path using LARS (sklearn's lasso_path)
# eps=1e-3 means the path goes from lambda_max down to lambda_max * eps
# n_alphas=200 controls the resolution of the path
alphas_lasso, coefs_lasso, _ = lasso_path(
    X_train.values,
    y_train.values,
    eps=1e-3,
    n_alphas=200,
    method='lars'   # LARS gives exact piecewise-linear path
)

# coefs_lasso shape: (n_features, n_alphas)
# Each row is one feature's coefficient trajectory as alpha decreases
print(f"Path computed: {len(alphas_lasso)} alpha values")
print(f"Alpha range: [{alphas_lasso.min():.4f}, {alphas_lasso.max():.4f}]")
print(f"Coefficient matrix shape: {coefs_lasso.shape}")

# Compute fraction of max alpha (easier to read on x-axis)
alpha_frac = alphas_lasso / alphas_lasso.max()

In [ ]:
# Plot the Lasso regularisation path
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left panel: coefficient trajectories vs log(alpha)
ax = axes[0]
for i, (coef_path, name) in enumerate(zip(coefs_lasso, feature_names)):
    ax.plot(np.log10(alphas_lasso), coef_path, color=COLORS[i], linewidth=2, label=name)

ax.axhline(0, color='black', linewidth=0.8, linestyle='--', alpha=0.5)
ax.set_xlabel('log₁₀(λ)', fontsize=12)
ax.set_ylabel('Coefficient', fontsize=12)
ax.set_title('Lasso Regularisation Path', fontsize=14)
ax.legend(loc='upper left', fontsize=8, ncol=2)
# Read right to left: large lambda (left) = empty model, small lambda (right) = full model
ax.invert_xaxis()

# Right panel: number of nonzero coefficients vs alpha fraction
ax = axes[1]
n_nonzero = np.sum(coefs_lasso != 0, axis=0)
ax.step(np.log10(alphas_lasso), n_nonzero, where='post', color='steelblue', linewidth=2)
ax.set_xlabel('log₁₀(λ)', fontsize=12)
ax.set_ylabel('Number of Selected Features', fontsize=12)
ax.set_title('Model Sparsity Along the Path', fontsize=14)
ax.set_yticks(range(len(feature_names) + 1))
ax.invert_xaxis()

# Mark where each feature enters (first becomes nonzero)
for i, (coef_path, name) in enumerate(zip(coefs_lasso, feature_names)):
    entry_idx = np.where(coef_path != 0)[0]
    if len(entry_idx) > 0:
        first_entry = entry_idx[0]
        ax.annotate(
            name[:4],  # shortened label
            xy=(np.log10(alphas_lasso[first_entry]), n_nonzero[first_entry]),
            fontsize=7, color=COLORS[i], ha='right'
        )

plt.tight_layout()
plt.savefig('lasso_path.png', dpi=100, bbox_inches='tight')
plt.show()

# Print feature entry order (importance rank by LARS)
print("\nFeature entry order (most important first):")
entry_alphas = {}
for i, (coef_path, name) in enumerate(zip(coefs_lasso, feature_names)):
    entry_idx = np.where(coef_path != 0)[0]
    if len(entry_idx) > 0:
        entry_alphas[name] = alphas_lasso[entry_idx[0]]

for rank, (name, alpha) in enumerate(
    sorted(entry_alphas.items(), key=lambda x: -x[1]), 1
):
    print(f"  {rank}. {name} (enters at λ={alpha:.4f})")

## 3. Comparing Lasso vs ElasticNet Paths

Now we create a dataset with highly correlated features to expose Lasso's weakness: among a group of correlated predictors, Lasso arbitrarily selects one and zeroes out the rest. ElasticNet, via its L2 component, distributes the weight across the group.

In [ ]:
# Construct a dataset where Lasso's failure is clear
# True model: y depends on x1 and x2 (correlated pair) plus x5 (independent)
rng = np.random.default_rng(42)
n = 300

# Independent base signals
z1 = rng.normal(0, 1, n)  # underlies x1, x2, x3 (correlated group)
z2 = rng.normal(0, 1, n)  # independent feature
noise_y = rng.normal(0, 0.5, n)

# Features: x1, x2, x3 are highly correlated (r ~ 0.95)
# x4, x5 are independent; x5 is truly predictive
X_corr = pd.DataFrame({
    'x1': z1 + 0.2 * rng.normal(0, 1, n),   # correlated group, TRUE predictor
    'x2': z1 + 0.2 * rng.normal(0, 1, n),   # correlated group, also TRUE predictor
    'x3': z1 + 0.2 * rng.normal(0, 1, n),   # correlated group, TRUE predictor
    'x4': rng.normal(0, 1, n),              # independent, NULL
    'x5': z2 + 0.1 * rng.normal(0, 1, n),   # independent, TRUE predictor
    'x6': rng.normal(0, 1, n),              # independent, NULL
})

# True model: y = 2*z1 + 3*z2 + noise (equivalently, ~2*x1 + 3*x5)
y_corr = 2 * z1 + 3 * z2 + noise_y

# Standardise
X_corr_scaled = pd.DataFrame(
    StandardScaler().fit_transform(X_corr),
    columns=X_corr.columns
)

# Check correlations
print("Feature correlations (relevant features are x1,x2,x3 and x5):")
print(X_corr_scaled.corr().round(2))

In [ ]:
# Compute both Lasso and ElasticNet paths on the correlated dataset
alphas_lasso_c, coefs_lasso_c, _ = lasso_path(
    X_corr_scaled.values, y_corr, eps=1e-3, n_alphas=200
)

# ElasticNet path with l1_ratio=0.5 (balanced L1/L2)
alphas_enet_c, coefs_enet_c, _ = enet_path(
    X_corr_scaled.values, y_corr, eps=1e-3, n_alphas=200, l1_ratio=0.5
)

corr_features = X_corr.columns.tolist()
COLORS_6 = ['#e41a1c', '#ff7f00', '#e6ab02', '#4daf4a', '#377eb8', '#984ea3']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, coefs, alphas, title in [
    (axes[0], coefs_lasso_c, alphas_lasso_c, 'Lasso Path\n(correlated features)'),
    (axes[1], coefs_enet_c, alphas_enet_c, 'ElasticNet Path (l1_ratio=0.5)\n(correlated features)'),
]:
    for i, (coef_path, name) in enumerate(zip(coefs, corr_features)):
        ls = '-' if name in ['x1', 'x2', 'x3', 'x5'] else '--'
        lw = 2.5 if name in ['x1', 'x2', 'x3', 'x5'] else 1.5
        ax.plot(np.log10(alphas), coef_path, color=COLORS_6[i],
                linewidth=lw, linestyle=ls, label=name)

    ax.axhline(0, color='black', linewidth=0.8, linestyle=':', alpha=0.5)
    ax.set_xlabel('log₁₀(λ)', fontsize=12)
    ax.set_ylabel('Coefficient', fontsize=12)
    ax.set_title(title, fontsize=13)
    ax.legend(fontsize=10)
    ax.invert_xaxis()

plt.suptitle(
    'Solid lines = true predictors (x1,x2,x3 correlated group + x5)',
    fontsize=11, y=1.02
)
plt.tight_layout()
plt.savefig('lasso_vs_enet_correlated.png', dpi=100, bbox_inches='tight')
plt.show()

print("\nObservation:")
print("  Lasso: among x1/x2/x3 (correlated group), selects approximately ONE.")
print("  ElasticNet: distributes weight across all three — grouping effect.")

## 4. Cross-Validated Lambda Selection

`LassoCV` and `ElasticNetCV` find the optimal $\lambda$ using $k$-fold cross-validation. We also apply the **one-standard-error rule**: select the most regularised model whose CV error is within one SE of the minimum. This produces a sparser model when performance is statistically similar.

In [ ]:
# LassoCV on the California Housing dataset
lasso_cv = LassoCV(
    cv=10,
    n_alphas=100,
    max_iter=10000,
    random_state=42,
    eps=1e-3
)
lasso_cv.fit(X_train.values, y_train.values)

# lasso_cv.alphas_: the grid of alpha values (decreasing)
# lasso_cv.mse_path_: CV MSE for each fold at each alpha, shape (n_alphas, n_folds)
mean_mse = lasso_cv.mse_path_.mean(axis=1)
std_mse = lasso_cv.mse_path_.std(axis=1)

optimal_idx = np.argmin(mean_mse)
optimal_alpha = lasso_cv.alphas_[optimal_idx]

# One-standard-error rule: largest alpha with MSE <= min_MSE + 1 SE
one_se_threshold = mean_mse[optimal_idx] + std_mse[optimal_idx]
one_se_candidates = np.where(mean_mse <= one_se_threshold)[0]
one_se_idx = one_se_candidates[0]  # largest alpha (most regularised) that qualifies
one_se_alpha = lasso_cv.alphas_[one_se_idx]

print(f"Optimal lambda (min CV-MSE):     {optimal_alpha:.5f}")
print(f"One-SE lambda (sparser model):   {one_se_alpha:.5f}")
print(f"LassoCV selected alpha:          {lasso_cv.alpha_:.5f}")

# Plot the CV error curve
fig, ax = plt.subplots(figsize=(9, 4))
ax.semilogx(lasso_cv.alphas_, mean_mse, color='steelblue', linewidth=2, label='CV MSE')
ax.fill_between(
    lasso_cv.alphas_,
    mean_mse - std_mse,
    mean_mse + std_mse,
    alpha=0.25, color='steelblue', label='±1 std'
)
ax.axvline(optimal_alpha, color='red', linestyle='--', linewidth=1.5, label=f'Min CV-MSE (λ={optimal_alpha:.4f})')
ax.axvline(one_se_alpha, color='orange', linestyle='--', linewidth=1.5, label=f'1-SE rule (λ={one_se_alpha:.4f})')
ax.set_xlabel('λ (log scale)', fontsize=12)
ax.set_ylabel('CV MSE', fontsize=12)
ax.set_title('LassoCV: Cross-Validated Error Curve', fontsize=14)
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig('lasso_cv_curve.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# ElasticNetCV: also searches over l1_ratio
enet_cv = ElasticNetCV(
    l1_ratio=[0.1, 0.3, 0.5, 0.7, 0.9, 0.95, 1.0],  # 1.0 = pure Lasso
    cv=10,
    n_alphas=100,
    max_iter=10000,
    random_state=42,
    eps=1e-3
)
enet_cv.fit(X_train.values, y_train.values)

print("ElasticNetCV Results:")
print(f"  Optimal l1_ratio: {enet_cv.l1_ratio_:.2f}")
print(f"  Optimal alpha:    {enet_cv.alpha_:.5f}")

# Compare selected features between LassoCV and ElasticNetCV
lasso_selected = np.where(lasso_cv.coef_ != 0)[0]
enet_selected = np.where(enet_cv.coef_ != 0)[0]

print(f"\nLassoCV selected {len(lasso_selected)} features: {[feature_names[i] for i in lasso_selected]}")
print(f"ElasticNetCV selected {len(enet_selected)} features: {[feature_names[i] for i in enet_selected]}")

# Evaluate on test set
for name, model in [('LassoCV', lasso_cv), ('ElasticNetCV', enet_cv)]:
    y_pred = model.predict(X_test.values)
    r2 = r2_score(y_test, y_pred)
    rmse = mean_squared_error(y_test, y_pred, squared=False)
    n_sel = np.sum(model.coef_ != 0)
    print(f"{name}: R²={r2:.4f}, RMSE={rmse:.4f}, features={n_sel}")

## 5. Feature Stability Along the Path

A robust feature is one that remains selected across a wide range of $\lambda$ values. We can compute each feature's **selection fraction** — the proportion of path values for which the coefficient is nonzero. This is a simpler version of stability selection.

In [ ]:
# Feature selection fraction across the Lasso path
# For each feature: fraction of lambda values where it is selected
selection_fraction = (coefs_lasso != 0).mean(axis=1)

# Sort by selection fraction
order = np.argsort(selection_fraction)[::-1]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Bar chart: selection fraction
ax = axes[0]
bars = ax.barh(
    [feature_names[i] for i in order],
    selection_fraction[order],
    color=[COLORS[i] for i in order]
)
ax.axvline(0.5, color='red', linestyle='--', linewidth=1.5, label='50% threshold')
ax.set_xlabel('Selection Fraction', fontsize=12)
ax.set_title('Feature Selection Fraction\nAcross Lasso Path', fontsize=13)
ax.legend(fontsize=10)
ax.set_xlim(0, 1.05)

# Scatter: selection fraction vs final coefficient magnitude
ax = axes[1]
final_coef = np.abs(coefs_lasso[:, -1])  # coefficient at smallest lambda
for i, name in enumerate(feature_names):
    ax.scatter(selection_fraction[i], final_coef[i], color=COLORS[i], s=120, label=name, zorder=3)
    ax.annotate(name, (selection_fraction[i], final_coef[i]),
                fontsize=9, xytext=(5, 3), textcoords='offset points')

ax.set_xlabel('Selection Fraction (path stability)', fontsize=12)
ax.set_ylabel('|Coefficient| at λ_min', fontsize=12)
ax.set_title('Stability vs Coefficient Magnitude', fontsize=13)
ax.legend(fontsize=8, loc='upper left', ncol=2)

plt.tight_layout()
plt.savefig('feature_stability.png', dpi=100, bbox_inches='tight')
plt.show()

print("\nFeature stability summary:")
for i in order:
    print(f"  {feature_names[i]:20s}: selection fraction = {selection_fraction[i]:.2f}")

## Exercise 5.1: Lasso Failure on Correlated Features

**Task:** Using the correlated dataset (`X_corr_scaled`, `y_corr`) created in Section 3:

1. Fit `LassoCV` and record which features are selected
2. Fit `ElasticNetCV` and record which features are selected
3. Compute Kendall's tau correlation between `x1` and `x2` to confirm their correlation
4. Print whether Lasso selected all three correlated features (`x1`, `x2`, `x3`) or just one

**Expected result:** Lasso selects only 1–2 from the `{x1, x2, x3}` group. ElasticNet selects all three.

In [ ]:
# YOUR CODE HERE
# ---------------
# 1. Fit LassoCV on X_corr_scaled, y_corr

# 2. Fit ElasticNetCV on X_corr_scaled, y_corr

# 3. Print selected features for each method

# 4. Print how many of {x1, x2, x3} are selected by each method

In [ ]:
# AUTO-GRADED TESTS — do not modify
def test_exercise_51():
    from sklearn.linear_model import LassoCV, ElasticNetCV
    
    lasso = LassoCV(cv=5, n_alphas=50, max_iter=5000, random_state=42)
    lasso.fit(X_corr_scaled.values, y_corr)
    enet = ElasticNetCV(l1_ratio=[0.5, 0.7, 0.9], cv=5, n_alphas=50, max_iter=5000, random_state=42)
    enet.fit(X_corr_scaled.values, y_corr)
    
    lasso_corr_count = sum(lasso.coef_[i] != 0 for i in [0, 1, 2])
    enet_corr_count = sum(enet.coef_[i] != 0 for i in [0, 1, 2])
    
    # Lasso should select fewer from the correlated group
    assert lasso_corr_count <= 2, (
        f"Expected Lasso to select <= 2 from correlated group, got {lasso_corr_count}. "
        "This demonstrates Lasso's arbitrary selection among correlated features."
    )
    # ElasticNet should select all three (grouping effect)
    assert enet_corr_count >= 2, (
        f"Expected ElasticNet to select >= 2 from correlated group, got {enet_corr_count}."
    )
    print(f"Lasso selected {lasso_corr_count}/3 correlated features.")
    print(f"ElasticNet selected {enet_corr_count}/3 correlated features.")
    print("Test passed: Lasso fails on correlated features; ElasticNet succeeds.")

test_exercise_51()

## Exercise 5.2: Interpret a Path

**Task:** Using `coefs_lasso` and `alphas_lasso` (computed on California Housing in Section 2):

1. Find the feature that **enters the path first** (at the largest $\lambda$ value) — this is the most important feature by Lasso ordering.
2. Find any feature that **changes sign** along the path (coefficient goes from positive to negative or vice versa).
3. Print both findings.

**Hint:** `entry_alphas` dict was computed in Section 2.

In [ ]:
# YOUR CODE HERE
# ---------------
# 1. Most important feature (enters first at largest lambda)

# 2. Features that change sign along the path

# 3. Print both

In [ ]:
# AUTO-GRADED TESTS — do not modify
def test_exercise_52():
    # First feature to enter: largest entry alpha
    most_important = max(entry_alphas.items(), key=lambda x: x[1])[0]
    assert most_important in feature_names, f"{most_important} not in feature names"
    
    # Sign-changing features: find features with both positive and negative coefficients
    sign_changers = []
    for i, name in enumerate(feature_names):
        path = coefs_lasso[i]
        nonzero = path[path != 0]
        if len(nonzero) > 1:
            if np.any(nonzero > 0) and np.any(nonzero < 0):
                sign_changers.append(name)
    
    print(f"Most important feature (enters first): {most_important}")
    print(f"Sign-changing features: {sign_changers if sign_changers else 'None found (normal for this dataset)'}")
    print("Test passed.")

test_exercise_52()

## Summary

### Key Takeaways

1. **LARS computes the complete Lasso path** in one pass — piecewise linear segments between knots where features enter or leave the active set.
2. **The entry order is an importance ranking** — features entering at large $\lambda$ have higher marginal correlation with the target given what the model has already fit.
3. **Lasso fails on correlated features** — among `{x1, x2, x3}` (r≈0.95), it selects at most one or two arbitrarily. ElasticNet's grouping effect distributes weight across all three.
4. **LassoCV selects $\lambda$ for prediction** — the one-SE rule additionally enforces sparsity when comparable performance is achievable with fewer features.
5. **Selection fraction** (fraction of path where a feature is selected) is a simple stability metric — a precursor to full stability selection.

### What's Next

Notebook 02 implements **stability selection** — converting the binary Lasso selection into a probability by running the path on many bootstrap subsamples. This provides the theoretical false discovery rate control that a single Lasso path cannot.

### Additional Resources
- [sklearn.linear_model.lasso_path](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.lasso_path.html) — LARS path computation
- Efron et al. (2004) "Least Angle Regression" — original LARS paper
- Zou & Hastie (2005) "Regularization via the Elastic Net" — grouping effect proof